### Tokenizer 

This section uses the SentencePiece tokenizer for making an extended tokenizer for the respective languages that's not in the current supported language list of M2M100.

In [1]:
import sentencepiece as spm
import os
# file,language_code 
os.makedirs('vocabs')
dataset_lang_corpora = [
    #[("dataset/cleaned-txt/bicol_cleaned.sentences.txt","bcl"),
    #("dataset/cleaned-txt/chavacano_cleaned.sentences.txt","chv"),
    #("dataset/cleaned-txt/bantoanon_cleaned.sentences.txt","btn")],
    #("dataset/cleaned-txt/cebuano_cleaned.sentences.txt","ceb"),
    #("dataset/cleaned-txt/english_cleaned.sentences.txt","en"),
    #("dataset/cleaned-txt/filipino-tl_cleaned.sentences.txt","tl"),
    #("dataset/cleaned-txt/ilokano_cleaned.sentences.txt","ilk"),
    [("dataset/cleaned-txt/ilonggo_cleaned.sentences.txt","ilg")
    #("dataset/cleaned-txt/ivatan_cleaned.sentences.txt","ivt"),
    #("dataset/cleaned-txt/kinaray-a_cleaned.sentences.txt","kry")],
    ],
    #[("dataset/cleaned-txt/manobo_cleaned.sentences.txt","mbo"),
    #("dataset/cleaned-txt/masbatenyo_cleaned.sentences.txt","mbyo"),
    #("dataset/cleaned-txt/pampanga_cleaned.sentences.txt","pam")],
    #[("dataset/cleaned-txt/pangasinan_cleaned.sentences.txt","pgn"),
    #("dataset/cleaned-txt/romblomanon_cleaned.sentences.txt","rblo"),
    #("dataset/cleaned-txt/sambal_cleaned.sentences.txt","sbl")],
    #("dataset/cleaned-txt/spanish_cleaned.sentences.txt","spa"),
    #[("dataset/cleaned-txt/tausug_cleaned.sentences.txt","tsg"),
    #("dataset/cleaned-txt/waray_cleaned.sentences.txt","wry"),
    #("dataset/cleaned-txt/yakan_cleaned.sentences.txt","ykn")],
    #[("dataset/cleaned-txt/yami_cleaned.sentences.txt","ymi")]

]


In [2]:
import math
def train_tokenizer(input_file,lang_code,vocab_size=3000,char_coverage=0.9995,model_type="bpe"):
    model_prefix = "vocabs/"+lang_code+"-spe"
    print(model_prefix)
    spm.SentencePieceTrainer.train(input=input_file,
                                   model_prefix=model_prefix,
                                   character_coverage=char_coverage,
                                   vocab_size=vocab_size,
                                   model_type=model_type,                                   
                                   )

def get_vocab_size(file_path):
    vocab = set()
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            tokens = line.strip().split(sep=' ')
            vocab.update(tokens)
    return len(vocab)


for i in range(0,len(dataset_lang_corpora)):
    for path,code in dataset_lang_corpora[i]:
        size = round(get_vocab_size(path)*0.1)
        print(size)
        train_tokenizer(path,code,vocab_size=(size))

2224
vocabs/ilg-spe


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: dataset/cleaned-txt/ilonggo_cleaned.sentences.txt
  input_format: 
  model_prefix: vocabs/ilg-spe
  model_type: BPE
  vocab_size: 2224
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
 

### Extending M2M100 1.2B Tokenizer


In [3]:
from transformers import M2M100Tokenizer, M2M100ForConditionalGeneration
import json
model_name = "facebook/m2m100_1.2B"
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

model.generation_config.max_length = 200
model.generation_config.early_stopping = True
model.generation_config.num_beams = 5
main_tokenizer = M2M100Tokenizer.from_pretrained(model_name)
print("Original Length:"+str(len(main_tokenizer)))
# sentence piece generates a .vocab file instead of a .json format
def spm_to_json_vocab(spm_model_path, json_vocab_path):
    sp = spm.SentencePieceProcessor()
    sp.load(spm_model_path)

    vocab = { sp.id_to_piece(i): i for i in range(sp.get_piece_size()) }

    with open(json_vocab_path, "w") as f:
        json.dump(vocab, f, ensure_ascii=False, indent=2)


def extend_model_with_lang_code(lang_code):
    spm_to_json_vocab("vocabs/"+lang_code+"-spe.model","vocabs/"+lang_code+"-spe.json")
    new_tokenizer = M2M100Tokenizer(spm_file="vocabs/"+lang_code+"-spe.model",vocab_file="vocabs/"+lang_code+"-spe.json")
    total_tokens_added = main_tokenizer.add_tokens(list(new_tokenizer.get_vocab().keys()))
    print(f"{lang_code}: added {total_tokens_added} tokens")



for i in range(0,len(dataset_lang_corpora)):
    for path,lang_code in dataset_lang_corpora[i]:
        extend_model_with_lang_code(lang_code)

model.resize_token_embeddings(len(main_tokenizer),mean_resizing=False)

#os.makedirs("tokenizers")
print(len(main_tokenizer))
main_tokenizer.save_pretrained("tokenizers/extended_m2m100tokenizer")
model.save_pretrained("models/extended_lang_m2m100_1.2B")
print("saved")



Original Length:128104
ilg: added 1320 tokens
129424


/usr/local/lib/python3.12/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


saved
